In [ ]:
import pandas as pd
from csae_pyutils import gsheet_to_df
from tqdm import tqdm
from apis_core.apis_entities.models import Work, Event, Place
from apis_core.apis_metainfo.models import Collection, TempEntityClass, Uri
from apis_core.apis_relations.models import EventWork, PlaceEvent
from apis_core.apis_vocabularies.models import EventWorkRelation, EventType, PlaceEventRelation

In [ ]:
col = Collection.objects.get(id=84)
domain = "burgtheater"
event_type = EventType.objects.get(id=1200)
event_work_relation = EventWorkRelation.objects.get(id=1201)
place_event_relation = PlaceEventRelation.objects.get(id=1202)
col, domain, event_work_relation, event_type, place_event_relation

In [ ]:
df = gsheet_to_df("10aUwBZzOnwUnTK6CwsOYliMYN6M2DK0Y8pluMs2kmHQ")

In [ ]:
works = {}
places = {}
start = 17000
for i, row in tqdm(df.iloc[start:].iterrows(), total=len(df) - start):
    year, month, day = row["date"].split("-")
    date_title = f"{int(day)}.{int(month)}.{year}"
    if not pd.isna(row["subtitle"]):
        title = f"Aufführung von {row["title"]} {row["subtitle"]}, {date_title}"
    else:
        title = f"Aufführung von {row["title"]}, {date_title}"
    try:
        work = works[row["work_id"]]
    except KeyError:
        work = Uri.objects.get(uri=row["work_id"]).entity
        work = Work.objects.get(id=work.id)
        works[row["work_id"]] = work
    if pd.isna(row["pmb_place_id"]):
        place = False
    else:
        try:
            place = places[int(row["pmb_place_id"])]
        except KeyError:
            place = Place.objects.get(id=int(row["pmb_place_id"]))
            places[int(row["pmb_place_id"])] = place
    event_id = row["burgtheater_event_url"]
    pmb_uri, _ = Uri.objects.get_or_create(uri=event_id, domain=domain)
    entity = pmb_uri.entity
    if entity:
        entity.collection.add(col)
        entity = Event.objects.get(id=entity.id)
    else:
        entity = Event.objects.create(
            name=title,
            start_date_written=row["date"],
            end_date_written=row["date"],
            kind=event_type
        )
        entity.collection.add(col)
        pmb_uri.entity = entity
        pmb_uri.save()
    EventWork.objects.get_or_create(
        related_event=entity,
        related_work=work,
        relation_type=event_work_relation,
        start_date_written=row["date"],
        end_date_written=row["date"]
    )
    if place:
        PlaceEvent.objects.get_or_create(
            related_place=place,
            related_event=entity,
            start_date_written=row["date"],
            end_date_written=row["date"],
            relation_type=place_event_relation
        )
    

    
    